In [ ]:
import sys; sys.path.append('..')
import MeshFEM, mesh, mesh_energy, param_utils, benchmark, viewer
import numpy as np
import sim_utils, param_utils

import matplotlib
from matplotlib import pyplot as plt
import visualization

import newton_flow, newton_flow_utils, py_newton_optimizer
from Stretch2Relax import extra_utils

# Load Meshes and compute Tutte embedding

In [ ]:
m_rest = mesh.Mesh('../Stretch2Relax/ToysMesh/Hilbert_opt_2d.obj')
m_defo = mesh.Mesh('../Stretch2Relax/ToysMesh/Hilbert_init_2d.obj')
v = mesh_energy.NodalVars(m_rest, 2)
v.setVars(m_defo.vertices().ravel())

In [ ]:
nf = newton_flow.symmetric_dirichlet(m_rest, v)
prob = py_newton_optimizer.NewtonMultiobjectiveProblem(v, [nf])

In [ ]:
# Nullspace pinning strategy
# TODO: add epsilon * low rank.
FIX_VARS = False
if FIX_VARS:
    fv = sim_utils.getBBoxVars(m_rest, sim_utils.BBoxFace.MIN_X)
    prob.setFixedVars(fv)
else:
    # nf.elementHessianShift = 1e-8
    prob.hessianShift = 1e-10
    prob.useRelativeHessianShift = False

In [ ]:
always_project = True

In [ ]:
opt = prob.optimizer()
opt.options.hessianProjectionController.startWithProjectionActive = False
opt.options.hessianProjectionController.numProjectionStepsBeforeDisable = 1
opt.options.hessianProjectionController.numConsecutiveIndefiniteStepsBeforeEnable = 0
if always_project: opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAlways()

## Possion-based Extrapolate

In [ ]:
m_rest_3 = mesh.Mesh('../Stretch2Relax/ToysMesh/Hilbert_opt_2d.obj', embeddingDimension=3)
param_poisson, prob_poisson = extra_utils.getParamProb(m_rest_3, m_defo.vertices(), FIX_VARS=FIX_VARS)

opt2 = prob_poisson.optimizer()
opt2.options = opt.options

In [ ]:
param_poisson.elementHessianShift = nf.elementHessianShift
prob_poisson.hessianShift = prob.hessianShift
prob_poisson.useRelativeHessianShift = prob.useRelativeHessianShift

### For NewtonFlow Baseline comparisons

In [ ]:
def eval_linear_extrapolate(x_0, coeffs, alphas):
    return np.array([(x_0 + coeffs[0] * a).reshape(-1, 2) for a in alphas])

def eval_poisson_based_extrapolate(x_0, coeffs, alphas):    
    prob_poisson.setVars(x_0)
    uvs_extra = []
    c_0 = x_0.reshape(-1,2).mean(axis=0)
    for a in alphas:
        uv_new = extra_utils.paramNewtonstepExtrapolation(m_rest_3, coeffs[0], param_poisson, a, Linv, method = 'Eulerian', fixedVind=None, fixedUV=m_defo.vertices()[90])
        uv_new += c_0 - uv_new.mean(axis=0)
        uvs_extra.append(uv_new)
    return np.array(uvs_extra)

In [ ]:
import newton_flow_utils as nfu
fv = nfu.ground_truth_flow(opt, 1.0, verbose=True, grad_tol=1e-8)

## Method Comparison

In [ ]:
always_project = True
opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAdaptive()
opt.options.hessianProjectionController.startWithProjectionActive = False
opt.options.hessianProjectionController.numConsecutiveIndefiniteStepsBeforeEnable = 0
opt.options.hessianProjectionController.numProjectionStepsBeforeDisable = 1
if always_project: opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAlways()

In [ ]:
import importlib
importlib.reload(extra_utils);

In [ ]:
extrapolation_dist = 10
constant_speed = False
num_frames = min(500, len(fv))

methods = [(1, nfu.eval_trajectory_taylor, 'Newton'),
           (2, nfu.eval_trajectory_taylor, 'Deg 2 Taylor'),
           (3, nfu.eval_trajectory_taylor, 'Deg 3 Taylor'),
           (1, extra_utils.RotationStrainExtrapolation(prob_poisson), 'Poisson'),
           (14, nfu.eval_trajectory_vector_pade, 'Pade 14'),
           (19, nfu.eval_trajectory_vector_pade, 'Pade 19')
]

ff = lambda i:  visualization.flow_frame(i, opt, fv, extrapolation_dist, constant_speed,
                         extrapolation_method_list=methods, truncate=True, corners_only=True)

# visualization.writeVideo('Hilbert_Newton_Deg2Taylor_Poisson_compare.mp4', num_frames, ff, skipFrame=2)

In [ ]:
benchmark.reset()
ff(50)
benchmark.report()

In [ ]:
ff(200)

In [ ]:
ff = lambda i:  visualization.flow_frame(i, opt, fv, extrapolation_dist, constant_speed,
                         extrapolation_method_list=baseline_methods
                         + [(2, nfu.eval_trajectory_logspiral, 'Deg 2 Spiral')])
visualization.writeVideo('spiral_deg_2_compare.mp4', num_frames, ff)

ff = lambda i:  visualization.flow_frame(i, opt, fv, extrapolation_dist, constant_speed,
                         extrapolation_method_list=baseline_methods
                         + [(3, nfu.eval_trajectory_logspiral, 'Deg 3 Spiral')])
visualization.writeVideo('spiral_deg_3_compare.mp4', num_frames, ff)

extrapolation_dist = 5
ff = lambda i:  visualization.flow_frame(i, opt, fv, extrapolation_dist, constant_speed,
                         extrapolation_method_list=baseline_methods
                         + [(14, nfu.eval_trajectory_vector_pade, 'Deg 14 Pade')],
                         truncate=True)
visualization.writeVideo('pade_compare.mp4', num_frames, ff)